# Lab 13: LLMs for Code Generation and Bug Detection


In [4]:
!pip install groq


In [5]:
import os
from groq import Groq

groq_api_key = "api"
client = Groq(api_key=groq_api_key)

def ask_llm(prompt, system_msg="You are an expert software engineer.", model="llama-3.1-8b-instant"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content

print("Setup complete.")


Setup complete.


## Task 1: Code Generation from Natural Language


In [6]:
# Task 1a: Generate a sorting algorithm
prompt_sort = """
Write a Python function that implements merge sort.
Include:
- The function signature with type hints
- Docstring explaining the algorithm
- The implementation
- A test with sample input and output
"""

code = ask_llm(prompt_sort)
print(code)

```python
def merge_sort(arr: list[int]) -> list[int]:
    """
    This function implements the merge sort algorithm, a divide-and-conquer algorithm 
    that splits a list of elements into two halves, recursively sorts each half, 
    and then merges the two sorted halves.

    Args:
        arr (list[int]): The list of integers to be sorted.

    Returns:
        list[int]: The sorted list of integers.
    """

    # Base case: If the length of the array is 1 or less, return the array (since it's already sorted)
    if len(arr) <= 1:
        return arr

    # Find the middle index of the array
    mid = len(arr) // 2

    # Divide the array into two halves
    left_half = arr[:mid]
    right_half = arr[mid:]

    # Recursively sort each half
    left_half = merge_sort(left_half)
    right_half = merge_sort(right_half)

    # Merge the two sorted halves
    return merge(left_half, right_half)


def merge(left: list[int], right: list[int]) -> list[int]:
    """
    This function merges

In [7]:
# Task 1b: Generate a REST API endpoint
prompt_api = """
Write a Python Flask REST API with the following endpoints:
1. GET /students - returns a list of all students
2. POST /students - adds a new student
3. GET /students/<id> - returns a specific student by ID

Use an in-memory list to store students (no database needed).
Each student has: id, name, grade, subject
"""

api_code = ask_llm(prompt_api)
print(api_code)

**Student API using Flask**

Below is a simple implementation of a Flask REST API to manage students.

### Requirements

* Python 3.8+
* Flask 2.0+

### Code

```python
from flask import Flask, request, jsonify

app = Flask(__name__)

# In-memory list to store students
students = [
    {"id": 1, "name": "John Doe", "grade": 10, "subject": "Math"},
    {"id": 2, "name": "Jane Doe", "grade": 11, "subject": "Science"},
]

# GET /students - returns a list of all students
@app.route('/students', methods=['GET'])
def get_students():
    """Returns a list of all students."""
    return jsonify(students)

# POST /students - adds a new student
@app.route('/students', methods=['POST'])
def add_student():
    """Adds a new student."""
    data = request.json
    if 'id' not in data or 'name' not in data or 'grade' not in data or 'subject' not in data:
        return jsonify({"error": "Missing required fields"}), 400
    new_student = {
        "id": data['id'],
        "name": data['name'],
     

## Task 2: Bug Detection and Fixing


In [8]:
# Buggy code samples
buggy_codes = [
    {
        "description": "Fibonacci function with a bug",
        "code": """
def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]

    fib = [0, 1]
    for i in range(2, n):
        fib.append(fib[i-1] + fib[i-2])
    return fib

print(fibonacci(5))   # Expected: [0, 1, 1, 2, 3] but getting wrong output
"""
    },
    {
        "description": "Off-by-one error in binary search",
        "code": """
def binary_search(arr, target):
    left, right = 0, len(arr)
    while left < right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid
        else:
            right = mid
    return -1
"""
    },
    {
        "description": "SQL injection vulnerability",
        "code": """
import sqlite3

def get_user(username):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query = f"SELECT * FROM users WHERE username = '{username}'"
    cursor.execute(query)
    return cursor.fetchone()
"""
    }
]

for item in buggy_codes:
    print(f"=== {item['description']} ===")
    prompt = f"""
Analyze this Python code for bugs, errors, or security vulnerabilities.
For each issue found:
1. Identify the bug/issue
2. Explain why it's a problem
3. Provide the fixed version

Code:
{item['code']}
"""
    analysis = ask_llm(prompt)
    print(analysis)
    print("\n" + "="*60 + "\n")

=== Fibonacci function with a bug ===
**Bug Analysis**

### 1. Incorrect Fibonacci Sequence Generation

**Bug:** The Fibonacci sequence generation is incorrect.

**Why it's a problem:** The Fibonacci sequence is generated incorrectly because the code is trying to access the `fib` list with an index that is out of range. In Python, list indices start at 0, so when `i` is 2, `fib[i-1]` and `fib[i-2]` are trying to access the first and second elements of the list, which are `0` and `1` respectively. However, when `i` is 3, `fib[i-1]` and `fib[i-2]` are trying to access the second and third elements of the list, which are `1` and `0` respectively. This is incorrect because the Fibonacci sequence is generated by adding the previous two numbers, not the current and previous numbers.

**Fixed Version:**
```python
def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    
    fib = [0, 1]
    for i in range(2, n):
        fib.append(fib[i-2] + fib[i-1])  # Fix:

## Task 3: Code Refactoring


In [9]:
messy_code = """
def calc(a,b,c,d,e):
    x = a+b
    y = x*c
    z = y-d
    if z > 0:
        result = z/e
    else:
        result = 0
    print('the result is: ' + str(result))
    return result

# Calculate employee bonus
# a=base_salary, b=overtime_pay, c=performance_multiplier, d=deductions, e=tax_rate
calc(50000, 5000, 1.2, 3000, 1.3)
"""

refactor_prompt = f"""
Refactor the following Python code to make it:
1. More readable with meaningful variable/function names
2. Follow PEP 8 style guidelines
3. Include proper docstring
4. Use type hints
5. Handle edge cases properly

Original code:
{messy_code}
"""

refactored = ask_llm(refactor_prompt)
print(refactored)

Here's the refactored code:

```python
def calculate_employee_bonus(
    base_salary: float,
    overtime_pay: float,
    performance_multiplier: float,
    deductions: float,
    tax_rate: float
) -> float:
    """
    Calculates the employee bonus based on the provided parameters.

    Args:
        base_salary (float): The employee's base salary.
        overtime_pay (float): The employee's overtime pay.
        performance_multiplier (float): The performance multiplier.
        deductions (float): The deductions from the bonus.
        tax_rate (float): The tax rate applied to the bonus.

    Returns:
        float: The calculated employee bonus.

    Raises:
        ValueError: If any of the input parameters are negative.
    """

    # Validate input parameters
    if any(param < 0 for param in [base_salary, overtime_pay, performance_multiplier, deductions, tax_rate]):
        raise ValueError("Input parameters cannot be negative.")

    # Calculate the bonus
    total_pay = base

## Task 4: Code Translation (Python to JavaScript)


In [10]:
python_code = """
def find_duplicates(lst):
    seen = set()
    duplicates = []
    for item in lst:
        if item in seen:
            if item not in duplicates:
                duplicates.append(item)
        else:
            seen.add(item)
    return sorted(duplicates)

numbers = [1, 2, 3, 2, 4, 5, 3, 6, 1]
print(find_duplicates(numbers))  # Output: [1, 2, 3]
"""

translate_prompt = f"""
Translate this Python code to JavaScript (ES6+).
Preserve the same logic and functionality.
Add a brief comment explaining each major step.

Python code:
{python_code}
"""

js_code = ask_llm(translate_prompt)
print(js_code)

Here's the equivalent JavaScript code (ES6+) that preserves the same logic and functionality:

```javascript
/**
 * Finds and returns the duplicates in a given list, sorted in ascending order.
 * 
 * @param {Array} lst The input list to find duplicates in.
 * @returns {Array} A sorted list of duplicates found in the input list.
 */
function findDuplicates(lst) {
  // Create a Set to store unique items we've seen so far.
  const seen = new Set();
  // Create an array to store the duplicates we find.
  const duplicates = [];

  // Iterate over each item in the input list.
  lst.forEach((item) => {
    // If the item is already in the 'seen' Set, it's a duplicate.
    if (seen.has(item)) {
      // Add the item to the 'duplicates' array only if it's not already there.
      if (!duplicates.includes(item)) {
        duplicates.push(item);
      }
    } else {
      // If the item is not in the 'seen' Set, add it to the Set.
      seen.add(item);
    }
  });

  // Return the sorted list of 

## Task 5: Generate Unit Tests


In [11]:
function_to_test = """
def calculate_discount(price: float, discount_percent: float, min_price: float = 0) -> float:
    if price < 0:
        raise ValueError("Price cannot be negative")
    if not (0 <= discount_percent <= 100):
        raise ValueError("Discount must be between 0 and 100")
    discounted = price * (1 - discount_percent / 100)
    return max(discounted, min_price)
"""

test_prompt = f"""
Write comprehensive Python unit tests for this function using pytest.
Include tests for:
- Normal cases
- Edge cases (0%, 100% discount)
- Error cases (negative price, invalid discount)
- Minimum price constraint

Function:
{function_to_test}
"""

tests = ask_llm(test_prompt)
print(tests)

Here's an example of comprehensive Python unit tests for the `calculate_discount` function using pytest:

```python
# tests/test_discount_calculator.py

import pytest
from discount_calculator import calculate_discount  # Import the function to be tested

def test_normal_case():
    """Test normal case with a price and discount"""
    price = 100.0
    discount_percent = 20.0
    expected_result = 80.0
    assert calculate_discount(price, discount_percent) == expected_result

def test_zero_discount():
    """Test case with 0% discount"""
    price = 100.0
    discount_percent = 0.0
    expected_result = 100.0
    assert calculate_discount(price, discount_percent) == expected_result

def test_hundred_percent_discount():
    """Test case with 100% discount"""
    price = 100.0
    discount_percent = 100.0
    expected_result = 0.0
    assert calculate_discount(price, discount_percent) == expected_result

def test_negative_price():
    """Test case with negative price"""
    price = -100.0